## **Import & Setup**

In [1]:
import json, time, os, sys, gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm
from huggingface_hub import login

In [2]:
!nvidia-smi

Sun May 31 00:01:54 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 545.29.06              Driver Version: 545.29.06    CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A40                     Off | 00000000:CA:00.0 Off |                    0 |
|  0%   35C    P0              75W / 300W |  11694MiB / 46068MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [3]:
# Log in to HuggingFace:
login("hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

In [4]:
print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python version: 3.10.12 (main, Feb  4 2025, 14:57:36) [GCC 11.4.0]
PyTorch version: 2.7.0+cu126
CUDA available: True
GPU: NVIDIA A40


## **Help Functions**

In [5]:
def get_embedding(text):
    prompt = "Represent this sentence for retrieval: " + text
    inputs = emb_tokenizer(prompt, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        out = emb_model(**inputs)
        emb = F.normalize(out.last_hidden_state[:, 0], p=2, dim=1)
    return emb.cpu().numpy()[0]

In [6]:
def format_llm_prompt(top3_df, nace_code, nace_description, user_query=None):
    user_context = f'The user wants to open a business of type: "{user_query}"\n\n' if user_query else ""
    nace_info = f"**NACE Code: {nace_code} \u2014 {nace_description}**\n\n"
    block = "Based on data analysis, here are the top 3 recommended neighborhoods in Volos:\n\n"
    for i, row in top3_df.iterrows():
        block += f"{i+1}. **{row['Neighborhood']}**\n   {row['Description']}\n\n"
    instruction = (
        "Please present the 3 neighborhoods in a numbered list format like this:\n"
        "1. Neighborhood Name: Summary of strengths...\n"
        "2. Neighborhood Name: Summary of strengths...\n"
        "3. Neighborhood Name: Summary of strengths...\n\n"
        "Each point should be 2\u20134 sentences long, clearly explaining why the neighborhood "
        "is a good fit for the business. Avoid ranking or comparisons \u2014 describe each one "
        "positively and independently."
    )
    return user_context + nace_info + block + instruction

In [7]:
def generate_llama(prompt_text):
    messages = [{"role": "user", "content": prompt_text}]
    inputs = gen_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(gen_model.device)
    with torch.no_grad():
        out = gen_model.generate(
            **inputs, max_new_tokens=512, do_sample=False,
            pad_token_id=gen_tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return gen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [8]:
def rag_answer(query):
    q = get_embedding(query)
    
    # predict NACE class
    sims = cosine_similarity([q], nace_emb_matrix)[0]
    nace_code = nace_codes[int(np.argmax(sims))]
    nace_description = df_naces.loc[df_naces["NACE Code"] == nace_code, "Class Description"].values[0]
    
    # top-10 candidates for this NACE
    row = top10_df[top10_df["NACE Code"] == nace_code]
    top10_names = row.iloc[0, 1:].tolist() if not row.empty else []
    
    # rerank candidates by query<->description similarity, keep top-3
    cand = [(n, neigh_emb_lookup[n]) for n in top10_names if n in neigh_emb_lookup]
    desc_emb = np.stack([e for _, e in cand])
    rr = cosine_similarity([q], desc_emb)[0]
    order = np.argsort(rr)[::-1][:3]
    top3 = pd.DataFrame([{
        "Neighborhood": cand[k][0],
        "Description": neigh_desc_lookup.get(cand[k][0], ""),
    } for k in order]).reset_index(drop=True)
    
    prompt = format_llm_prompt(top3, nace_code, nace_description, user_query=query)
    return generate_llama(prompt)

## **Load Embedder & Generator**

In [9]:
GEN_MODEL_ID    = "meta-llama/Meta-Llama-3-8B-Instruct"
EMB_MODEL_ID    = "BAAI/bge-large-en-v1.5"

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

emb_tokenizer = AutoTokenizer.from_pretrained(EMB_MODEL_ID)
emb_model = AutoModel.from_pretrained(EMB_MODEL_ID).to(device)
emb_model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 1024, padding_idx=0)
    (position_embeddings): Embedding(512, 1024)
    (token_type_embeddings): Embedding(2, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-23): 24 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, 

In [12]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

In [13]:
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_ID)
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_ID,
    device_map="auto",
    quantization_config=quant_config,
)
gen_model.eval()

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((409

## **Load the Data**

In [14]:
NACE_CSV        = "../Files/RAG/unique_naces.csv"
TOP10_CSV       = "../Files/RAG/top10_final.csv"
NEIGH_DESC_CSV  = "../Files/RAG/neighborhood_descriptions_10.csv"
NACE_EMB_JSON   = "../Files/RAG/nace_descriptions_embeddings.json"
NEIGH_EMB_JSON  = "../Files/RAG/neighborhoods_descriptions_embeddings_10.json"
INFERENCE_JSONL = "../Files/inference_dataset.jsonl"

In [15]:
df_naces = pd.read_csv(NACE_CSV)
top10_df = pd.read_csv(TOP10_CSV)
neigh_desc_df = pd.read_csv(NEIGH_DESC_CSV)

with open(NACE_EMB_JSON, "r", encoding="utf-8") as f:
    precomputed_nace_embeddings = json.load(f)
with open(NEIGH_EMB_JSON, "r", encoding="utf-8") as f:
    precomputed_neigh_embeddings = json.load(f)

with open(INFERENCE_JSONL, "r", encoding="utf-8") as f:
    inference_data = [json.loads(line) for line in f]
print(f"{len(inference_data)} inference queries (should be 159, same as FT)")

159 inference queries (should be 159, same as FT)


In [16]:
nace_codes = [item["nace_code"] for item in precomputed_nace_embeddings]
nace_emb_matrix = np.array([item["embedding"] for item in precomputed_nace_embeddings])

neigh_emb_lookup  = {r["neighborhood"]: np.array(r["embedding"]) for r in precomputed_neigh_embeddings}
neigh_desc_lookup = dict(zip(neigh_desc_df["Neighborhood"], neigh_desc_df["Description"]))

## **Perform Inference**

In [17]:
# warm-up: absorbs CUDA kernel-compile cost, NOT timed
_ = rag_answer(inference_data[0]['prompt'])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [19]:
# run inference for all and measure time per query
results, per_q = [], []
for ex in tqdm(inference_data):
    qy = ex["prompt"]
    t0 = time.perf_counter()
    resp = rag_answer(qy)
    per_q.append(time.perf_counter() - t0)
    results.append({"prompt": qy, "expected_completion": ex["completion"], "model_output": resp})

print(f"queries={len(per_q)}  total={sum(per_q):.1f}s  "
      f"mean/q={np.mean(per_q):.2f}s  median/q={np.median(per_q):.2f}s")

100%|█████████████████████████████████████████████████████████████████████████████████████████| 159/159 [40:22<00:00, 15.24s/it]

queries=159  total=2422.4s  mean/q=15.24s  median/q=15.29s


In [ ]:
# save results
OUTPUT_JSONL = "../Files/inference_outputs_rag.jsonl"

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")